# Exploratory Notebook : Vector Database
## Introduction 

Now that we’ve completed the chunking and embedding steps, we can create our vector database. The idea is that this database stores the vectors we’ve created. Thanks to these vectors, we can search our documents based on their meaning, rather than just using keywords.                

To do this, we’ll use pgvector because:             
- it allows us to store all of our reference documents and vectors in a single database. They are linked together by keys, which makes it possible to quickly find matches
- it performs well with up to several million vectors, which is sufficient for our current volume
- it supports SQL queries, allowing us to search for meaning and retrieve documents in a single query

### 01. Docker compose 

So we'll create a Docker Compose file to ensure full reproducibility. It can be found in the `docker-compose.yml` file.

We use pgvector, which adds vector support to PostgreSQL

To do this, we'll use
-  the pgvector image name, which can be found here: [pgvector github](https://github.com/pgvector/pgvector) 
- the PostgreSQL credentials: [postgreSQL image documentation](https://hub.docker.com/_/postgres)
 
The credentials are in the `.env` file

## 02. PostgreSQL database 

We will now define the structure of our database. We will define three tables: 
- document: one row per PDF 
- chunk: one row per chunk, including its pages and vectors 
- query: one row per question asked, to keep track of the questions

The basic idea is that tables can be linked together using keys: 
- A chunk is linked to a document

We will also add constraints to protect the database 
- No duplicates if we reload data into the database
- If a document is deleted, its chunks are deleted along with it, so that there are no chunks without a document

The database has now been created, we can connect our notebook to it.

In [3]:
import os 
import psycopg
from dotenv import load_dotenv
from pgvector.psycopg import register_vector 
import numpy as np 
import pandas as pd 

To establish the connection, we'll use this documentation: [osycopg connection classes documentation](https://www.psycopg.org/psycopg3/docs/api/connections.html)

In [2]:
load_dotenv() 

conn = psycopg.connect(autocommit=True)
register_vector(conn)

Now that we have a connection to our database, we'll load our documents and our embeddings into it

In [5]:
df_chunks = pd.read_json("data/processed/chunks_01.json")
embeddings = np.load("data/processed/embeddings_01.npy")

# We retrieve the position of each chunk in its document and add it to our database
df_chunks["chunk_index"] = df_chunks.groupby("origin").cumcount()

All right, now we can insert our documents into the database
[pgvector insertion documentation](https://github.com/pgvector/pgvector-python)

Let's start by filling in the table with the names of the documents 

In [7]:
for name in df_chunks["origin"].unique():
    conn.execute(
        "INSERT INTO document (name) VALUES (%s) ON CONFLICT (name) DO NOTHING",
        (str(name),),
    )

print(conn.execute("SELECT id, name FROM document ORDER BY id").fetchall())

[(1, 'avis_ASNR_2026.pdf'), (2, 'avis_expertise_ASNR_2026.pdf'), (3, 'dossier_concertation_calogena_2026.pdf')]


and the chunk table

In [9]:
documents_ids = dict(conn.execute("SELECT name, id FROM document").fetchall())

for i, row in df_chunks.iterrows():
    conn.execute(
        """
        INSERT INTO chunk
            (document_id, chunk_index, starting_page, ending_page, 
            starting_character, ending_character, text, embedding)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        ON CONFLICT (document_id, chunk_index) DO NOTHING
        """,
        (documents_ids[row["origin"]],
         int(row["chunk_index"]),
         int(row["page_start"]),
         int(row["page_end"]),
         int(row["start"]),
         int(row["end"]),
         row["text"],
         embeddings[i],
        ),
    )

All right, now we can ask a question. It has to be converted into a vector itself. 